# Tugas Praktikum JS13 - Klasifikasi Angka Tulisan Tangan

Tugas ini membangun neural network untuk klasifikasi angka tulisan tangan dan membandingkan beberapa konfigurasi: jumlah neuron hidden layer, penambahan hidden layer, serta aktivasi Sigmoid dan ReLU.

TensorFlow/Keras dipakai jika tersedia. Karena TensorFlow belum kompatibel dengan Python 3.14 pada environment ini, notebook otomatis menggunakan dataset `load_digits` dari scikit-learn sebagai fallback lokal yang memiliki tugas klasifikasi angka yang sama.

In [5]:
import time
import numpy as np
import pandas as pd
from sklearn.datasets import load_digits
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler

try:
    from tensorflow.keras.datasets import mnist
    (X_train, y_train), (X_test, y_test) = mnist.load_data()
    dataset_name = "MNIST Keras"
    X_train = X_train.reshape(len(X_train), -1) / 255.0
    X_test = X_test.reshape(len(X_test), -1) / 255.0
except (ImportError, ModuleNotFoundError):
    digits = load_digits()
    X_train, X_test, y_train, y_test = train_test_split(
        digits.data / 16.0,
        digits.target,
        test_size=0.2,
        random_state=42,
        stratify=digits.target,
    )
    dataset_name = "Digits fallback (8x8)"

print(f"Dataset: {dataset_name}")
print(f"Bentuk data training: {X_train.shape}")
print(f"Bentuk data testing: {X_test.shape}")

Dataset: Digits fallback (8x8)
Bentuk data training: (1437, 64)
Bentuk data testing: (360, 64)


## Model dan evaluasi

Semua model menggunakan optimizer Adam melalui `MLPClassifier`, jumlah epoch maksimum yang sama, dan random state yang sama. Hal ini membuat perbandingan lebih adil.

In [6]:
def train_and_evaluate(name, hidden_layers, activation="relu"):
    model = MLPClassifier(
        hidden_layer_sizes=hidden_layers,
        activation=activation,
        solver="adam",
        batch_size=64,
        learning_rate_init=0.001,
        max_iter=120,
        early_stopping=True,
        validation_fraction=0.1,
        n_iter_no_change=8,
        random_state=42,
    )
    start = time.perf_counter()
    model.fit(X_train, y_train)
    training_time = time.perf_counter() - start
    predictions = model.predict(X_test)
    return {
        "Konfigurasi": name,
        "Hidden layer": str(hidden_layers),
        "Aktivasi": activation,
        "Akurasi": accuracy_score(y_test, predictions),
        "Loss akhir": model.loss_,
        "Waktu (detik)": training_time,
        "model": model,
        "predictions": predictions,
    }


experiments = [
    train_and_evaluate("Baseline 2 hidden", (128, 64), "relu"),
    train_and_evaluate("Neuron diperbesar", (256, 128), "relu"),
    train_and_evaluate("Tambah hidden layer", (128, 64, 32), "relu"),
    train_and_evaluate("Sigmoid", (128, 64), "logistic"),
    train_and_evaluate("ReLU", (128, 64), "relu"),
]

results = pd.DataFrame([
    {k: value for k, value in item.items() if k not in {"model", "predictions"}}
    for item in experiments
])
display(results.style.format({
    "Akurasi": "{:.2%}",
    "Loss akhir": "{:.6f}",
    "Waktu (detik)": "{:.3f}",
}))

,Konfigurasi,Hidden layer,Aktivasi,Akurasi,Loss akhir,Waktu (detik)
0,Baseline 2 hidden,"(128, 64)",relu,97.22%,0.020747,0.740
1,Neuron diperbesar,"(256, 128)",relu,98.06%,0.018406,1.244
2,Tambah hidden layer,"(128, 64, 32)",relu,97.78%,0.011041,0.813
3,Sigmoid,"(128, 64)",logistic,96.39%,0.102342,1.622
4,ReLU,"(128, 64)",relu,97.22%,0.020747,0.787


In [7]:
best_index = results["Akurasi"].idxmax()
best_result = experiments[best_index]
print("Model terbaik:", results.loc[best_index, "Konfigurasi"])
print(f"Akurasi: {results.loc[best_index, 'Akurasi']:.4f}")
print("\nClassification report model terbaik:")
print(classification_report(y_test, best_result["predictions"], zero_division=0))
print("Confusion matrix:")
print(confusion_matrix(y_test, best_result["predictions"]))

Model terbaik: Neuron diperbesar
Akurasi: 0.9806

Classification report model terbaik:
              precision    recall  f1-score   support

           0       1.00      0.97      0.99        36
           1       0.94      0.94      0.94        36
           2       1.00      1.00      1.00        35
           3       1.00      1.00      1.00        37
           4       0.95      1.00      0.97        36
           5       1.00      1.00      1.00        37
           6       1.00      0.97      0.99        36
           7       1.00      0.97      0.99        36
           8       0.92      0.94      0.93        35
           9       1.00      1.00      1.00        36

    accuracy                           0.98       360
   macro avg       0.98      0.98      0.98       360
weighted avg       0.98      0.98      0.98       360

Confusion matrix:
[[35  0  0  0  1  0  0  0  0  0]
 [ 0 34  0  0  0  0  0  0  2  0]
 [ 0  0 35  0  0  0  0  0  0  0]
 [ 0  0  0 37  0  0  0  0  0  0]
 [ 0

## Kesimpulan

Model dengan dua hidden layer menjadi baseline. Jumlah neuron dan jumlah layer tambahan dapat meningkatkan kapasitas model, tetapi konsekuensinya waktu pelatihan dapat bertambah. ReLU biasanya lebih cepat belajar dibanding Sigmoid pada hidden layer karena tidak mengalami saturasi separah Sigmoid. Konfigurasi terbaik ditentukan berdasarkan akurasi testing, dengan tetap memperhatikan loss dan waktu pelatihan.